In [ ]:
Seria para utilizar 
***csv
-tipo de viajero
-lugar de residencia
-lugar de residencia code
-territorio de las islas que viaja

***apis
moneda
***api 
clima 
lluvia

Hipotesis 1: Si el invierno europeo es frío, entonces Canarias recibe más turistas entre noviembre y marzo que en verano.
Hipotesis 2: Si llueve más en un mes, entonces llegan menos turistas 
Hipotesis 3: Cuales paises hacen mas turismo en las isla
hipotesis 4: Si en la pandemia estaba el espacio aereo cerrado, que turismo entro primero en las islas, nacionales o internacionales

In [1]:
import pandas as pd

df = pd.read_csv("dataset-ISTAC.csv", sep=",")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168300 entries, 0 to 168299
Data columns (total 15 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   TIME_PERIOD#es                   168300 non-null  str    
 1   TIME_PERIOD_CODE                 168300 non-null  str    
 2   TIPO_VIAJERO#es                  168300 non-null  str    
 3   TIPO_VIAJERO_CODE                168300 non-null  str    
 4   LUGAR_RESIDENCIA#es              168300 non-null  str    
 5   LUGAR_RESIDENCIA_CODE            168300 non-null  str    
 6   TERRITORIO#es                    168300 non-null  str    
 7   TERRITORIO_CODE                  168300 non-null  str    
 8   MEDIDAS#es                       168300 non-null  str    
 9   MEDIDAS_CODE                     168300 non-null  str    
 10  OBS_VALUE                        149441 non-null  float64
 11  CONFIDENCIALIDAD_OBSERVACION#es  0 non-null       float64
 12  NOTAS_OBSERVA

In [2]:
df["OBS_VALUE"].unique()

array([ 1.7120e+01,  4.7030e+01,  7.6900e+00, ...,  5.6780e+03,
        1.1888e+04, -2.7910e+03], shape=(54394,))

In [2]:
print(df.columns.tolist())

# valores distintos de cada columna que no sea numérica
for col in df.columns:
    if col != "OBS_VALUE":
        print(col, "->", df[col].nunique(), "valores distintos")

['TIME_PERIOD#es', 'TIME_PERIOD_CODE', 'TIPO_VIAJERO#es', 'TIPO_VIAJERO_CODE', 'LUGAR_RESIDENCIA#es', 'LUGAR_RESIDENCIA_CODE', 'TERRITORIO#es', 'TERRITORIO_CODE', 'MEDIDAS#es', 'MEDIDAS_CODE', 'OBS_VALUE', 'CONFIDENCIALIDAD_OBSERVACION#es', 'NOTAS_OBSERVACION', 'ESTADO_OBSERVACION#es', 'ESTADO_OBSERVACION_CODE']
TIME_PERIOD#es -> 187 valores distintos
TIME_PERIOD_CODE -> 187 valores distintos
TIPO_VIAJERO#es -> 3 valores distintos
TIPO_VIAJERO_CODE -> 3 valores distintos
LUGAR_RESIDENCIA#es -> 12 valores distintos
LUGAR_RESIDENCIA_CODE -> 12 valores distintos
TERRITORIO#es -> 5 valores distintos
TERRITORIO_CODE -> 5 valores distintos
MEDIDAS#es -> 5 valores distintos
MEDIDAS_CODE -> 5 valores distintos
CONFIDENCIALIDAD_OBSERVACION#es -> 0 valores distintos
NOTAS_OBSERVACION -> 1 valores distintos
ESTADO_OBSERVACION#es -> 1 valores distintos
ESTADO_OBSERVACION_CODE -> 1 valores distintos


In [3]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 28.05,
    "longitude": -16.57,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean,precipitation_sum,sunshine_duration",
    "timezone": "Atlantic/Canary",
}

r = requests.get(url, params=params)
r.raise_for_status()           # avisa si la petición falla
data = r.json()

clima = pd.DataFrame(data["daily"])
clima["time"] = pd.to_datetime(clima["time"])
clima["mes"] = clima["time"].dt.to_period("M")

mensual = clima.groupby("mes").agg(
    temp_media=("temperature_2m_mean", "mean"),
    lluvia_total=("precipitation_sum", "sum"),
)
print(mensual)

         temp_media  lluvia_total
mes                              
2021-01   16.854839         131.5
2021-02   16.875000          29.7
2021-03   17.480645          18.7
2021-04   18.773333          43.7
2021-05   20.716129           9.8
2021-06   21.963333          15.4
2021-07   25.264516           1.1
2021-08   25.629032           4.0
2021-09   24.740000           4.8
2021-10   23.180645           5.2
2021-11   19.450000          19.1
2021-12   18.732258          25.7
2022-01   18.022581          77.7
2022-02   18.703571          30.7
2022-03   16.809677          58.2
2022-04   18.663333          29.4
2022-05   21.374194           2.4
2022-06   22.130000           0.8
2022-07   26.445161           2.2
2022-08   26.074194           5.2
2022-09   23.896667         145.3
2022-10   23.580645           3.6
2022-11   22.313333           0.4
2022-12   19.883871          70.8
2023-01   18.122581           5.6
2023-02   15.967857          47.6
2023-03   22.096774           1.5
2023-04   21.1

In [26]:
import requests
import pandas as pd

frames = []

url = f"https://api.frankfurter.app/2020-01-01..2025-07-31"
r = requests.get(url, params={"from": "EUR", "to": "GBP"})
r.raise_for_status()
datos = r.json()["rates"]

df_anio = pd.DataFrame.from_dict(datos, orient="index")
frames.append(df_anio)

fx = pd.concat(frames)
fx.index = pd.to_datetime(fx.index)
fx = fx.rename(columns={"GBP": "eur_gbp"})

# Comprobación: filas por año
print(fx.groupby(fx.index.year).size())

# Agrupar por mes
fx["mes"] = fx.index.to_period("M")
fx_mensual = fx.groupby("mes")["eur_gbp"].mean().reset_index()

print(fx_mensual.head(20))

2019      1
2020    257
2021    258
2022    257
2023    255
2024    256
2025    148
dtype: int64
        mes   eur_gbp
0   2019-12  0.850800
1   2020-01  0.849273
2   2020-02  0.840946
3   2020-03  0.894595
4   2020-04  0.875469
5   2020-05  0.886853
6   2020-06  0.898781
7   2020-07  0.904667
8   2020-08  0.900809
9   2020-09  0.909474
10  2020-10  0.907411
11  2020-11  0.896053
12  2020-12  0.906244
13  2021-01  0.892665
14  2021-02  0.872683
15  2021-03  0.858732
16  2021-04  0.865268
17  2021-05  0.862583
18  2021-06  0.858720
19  2021-07  0.856130


In [5]:
df = df[["TIME_PERIOD#es","TIPO_VIAJERO#es","LUGAR_RESIDENCIA#es","LUGAR_RESIDENCIA_CODE","TERRITORIO#es","OBS_VALUE"]]

In [6]:
df = df.rename(columns={
    "TIME_PERIOD#es": "periodo",
    "TIPO_VIAJERO#es": "tipo _de_viajero",
    "LUGAR_RESIDENCIA#es": "residencia",
    "LUGAR_RESIDENCIA_CODE":"codigo_residencia",
    "TERRITORIO#es":"territorio_viaje",
    "OBS_VALUE":"numero_de_viajeros"
})

In [9]:
df.head(10)

,periodo,tipo _de_viajero,residencia,codigo_residencia,territorio_viaje,numero_de_viajeros
0,05/2011,Turista,Reino Unido,GB,Lanzarote,17.12
1,05/2011,Turista,Reino Unido,GB,Fuerteventura,47.03
2,05/2011,Turista,Reino Unido,GB,Gran Canaria,7.69
3,05/2011,Turista,Reino Unido,GB,Tenerife,28.27
4,05/2011,Turista,Reino Unido,GB,La Palma,-35.77
5,06/2011,Turista,Reino Unido,GB,Lanzarote,17.82
6,06/2011,Turista,Reino Unido,GB,Fuerteventura,8.22
7,06/2011,Turista,Reino Unido,GB,Gran Canaria,9.86
8,06/2011,Turista,Reino Unido,GB,Tenerife,10.44
9,06/2011,Turista,Reino Unido,GB,La Palma,-47.34


In [19]:
print(df.columns.tolist())

['periodo', 'tipo _de_viajero', 'residencia', 'codigo_residencia', 'territorio_viaje', 'numero_de_viajeros']


In [11]:
df.head()

,periodo,tipo _de_viajero,residencia,codigo_residencia,territorio_viaje,numero_de_viajeros
0,05/2011,Turista,Reino Unido,GB,Lanzarote,17.12
1,05/2011,Turista,Reino Unido,GB,Fuerteventura,47.03
2,05/2011,Turista,Reino Unido,GB,Gran Canaria,7.69
3,05/2011,Turista,Reino Unido,GB,Tenerife,28.27
4,05/2011,Turista,Reino Unido,GB,La Palma,-35.77


In [13]:
df["periodo"] = pd.to_datetime(df["periodo"], format="%m/%Y")

In [21]:
df_2020_2025 = df[df["periodo"].dt.year.between(2020, 2025)]

print(df_2020_2025.shape)
print(df_2020_2025["periodo"].min(), df_2020_2025["periodo"].max())

(60300, 6)
2020-01-01 00:00:00 2025-07-01 00:00:00


In [22]:
df_2020_2025

,periodo,tipo _de_viajero,residencia,codigo_residencia,territorio_viaje,numero_de_viajeros
520,2020-01-01,Turista,Reino Unido,GB,Lanzarote,-20.75
521,2020-01-01,Turista,Reino Unido,GB,Fuerteventura,-31.43
522,2020-01-01,Turista,Reino Unido,GB,Gran Canaria,3.17
523,2020-01-01,Turista,Reino Unido,GB,Tenerife,-3.30
524,2020-01-01,Turista,Reino Unido,GB,La Palma,-53.05
...,...,...,...,...,...,...
168215,2025-07-01,Turistas secundarios,Mundo (excluida España),5000_XES,Lanzarote,1245.00
168216,2025-07-01,Turistas secundarios,Mundo (excluida España),5000_XES,Fuerteventura,-2727.00
168217,2025-07-01,Turistas secundarios,Mundo (excluida España),5000_XES,Gran Canaria,-5036.00
168218,2025-07-01,Turistas secundarios,Mundo (excluida España),5000_XES,Tenerife,-132.00
